# Student CSV Task
## Bivek Thapa, 240495

In [ ]:
import os
os.environ["PYSPARK_PYTHON"] = "python"
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from pyspark.sql import functions as F

In [21]:
spark = SparkSession.builder.appName("DataPreprocessingStudent").master("local[*]").getOrCreate()

## Part A — Loading & Inspection

In [22]:
df = spark.read.csv("dataset/student.csv", header=True, inferSchema=True)

In [23]:
df.limit(5).show()

+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+
|student_id|         name|gender|age|      department|year| gpa|scholarship|     city|               email|
+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+
|       101| Aarav Sharma|  Male| 20|Computer Science|   2| 3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       102|  Priya Thapa|Female| 22|        Business|   4| 3.2|         No|  Pokhara| priya.thapa@uni.edu|
|       103|  Rohan Karki|  Male| 21|     Engineering|   3|NULL|         No| Lalitpur| rohan.karki@uni.edu|
|       104|     Sita Rai|Female| 19|Computer Science|   1| 3.9|        Yes|Kathmandu|    sita.rai@uni.edu|
|       105|Bikash Gurung|  Male| 23|        Business|   4| 2.7|         No|   Butwal|bikash.gurung@uni...|
+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+



In [24]:
from pyspark.sql.functions import col,sum as spark_sum,isnan,when

In [25]:
col_count = len(df.columns)
row_count = df.count()

print("Total number of columns:", col_count)
print("Total number of rows:", row_count)

Total number of columns: 10
Total number of rows: 25


In [26]:
schema = StructType([
    StructField("student_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("department", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("gpa", FloatType(), True),
    StructField("scholarship", StringType(), True),
    StructField("city", StringType(), True),
    StructField("email", StringType(), True),
])

In [27]:
df = spark.read.csv("dataset/student.csv", header=True, schema=schema)

## Part B — Null Value Handling

In [29]:
null_count = df.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns])
null_count.show()

+----------+----+------+---+----------+----+---+-----------+----+-----+
|student_id|name|gender|age|department|year|gpa|scholarship|city|email|
+----------+----+------+---+----------+----+---+-----------+----+-----+
|         0|   0|     0|  2|         1|   0|  3|          0|   0|    0|
+----------+----+------+---+----------+----+---+-----------+----+-----+



In [30]:
df.filter(col("gpa").isNull()).show()

+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+
|student_id|          name|gender|age| department|year| gpa|scholarship|    city|               email|
+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+
|       103|   Rohan Karki|  Male| 21|Engineering|   3|NULL|         No|Lalitpur| rohan.karki@uni.edu|
|       110|  Nisha Tamang|Female| 20|       Arts|   2|NULL|         No|Lalitpur|nisha.tamang@uni.edu|
|       122|Dipika Niroula|Female| 20|   Business|   2|NULL|         No|  Dharan|dipika.niroula@un...|
+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+



In [32]:
df_filled = df.fillna({
    "gpa": df.agg(F.avg("gpa")).first()[0],
    "department": "Undeclared",
    "age": 20
})
df_filled.show()

+----------+----------------+------+---+----------------+----+---------+-----------+---------+--------------------+
|student_id|            name|gender|age|      department|year|      gpa|scholarship|     city|               email|
+----------+----------------+------+---+----------------+----+---------+-----------+---------+--------------------+
|       101|    Aarav Sharma|  Male| 20|Computer Science|   2|      3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       102|     Priya Thapa|Female| 22|        Business|   4|      3.2|         No|  Pokhara| priya.thapa@uni.edu|
|       103|     Rohan Karki|  Male| 21|     Engineering|   3|3.3045454|         No| Lalitpur| rohan.karki@uni.edu|
|       104|        Sita Rai|Female| 19|Computer Science|   1|      3.9|        Yes|Kathmandu|    sita.rai@uni.edu|
|       105|   Bikash Gurung|  Male| 23|        Business|   4|      2.7|         No|   Butwal|bikash.gurung@uni...|
|       106|  Anita Shrestha|Female| 20|     Engineering|   2|      3.5|

## Part C — Selecting & Filtering

In [33]:
df.select("name", "department", "gpa").show()

+----------------+----------------+----+
|            name|      department| gpa|
+----------------+----------------+----+
|    Aarav Sharma|Computer Science| 3.8|
|     Priya Thapa|        Business| 3.2|
|     Rohan Karki|     Engineering|NULL|
|        Sita Rai|Computer Science| 3.9|
|   Bikash Gurung|        Business| 2.7|
|  Anita Shrestha|     Engineering| 3.5|
|   Deepak Pandey|            Arts| 3.1|
|     Kavya Joshi|Computer Science| 3.7|
|    Suresh Limbu|     Engineering| 2.9|
|    Nisha Tamang|            Arts|NULL|
|   Prakash Bista|        Business| 3.4|
|     Manisha Oli|Computer Science| 3.6|
| Rajesh Adhikari|     Engineering| 3.0|
|    Sunita Magar|            Arts| 2.8|
|    Kiran Poudel|Computer Science| 3.3|
|     Rupa Basnet|            NULL| 3.7|
|  Anil Chaudhary|     Engineering| 2.6|
|    Puja Koirala|        Business| 3.5|
|     Nabin Dahal|            Arts| 3.2|
|Kabita Bhattarai|Computer Science| 3.9|
+----------------+----------------+----+
only showing top

In [34]:
df.where((col("gpa") > 3.5) & (col("department") == "Computer Science")).show()

+----------+----------------+------+----+----------------+----+---+-----------+---------+--------------------+
|student_id|            name|gender| age|      department|year|gpa|scholarship|     city|               email|
+----------+----------------+------+----+----------------+----+---+-----------+---------+--------------------+
|       101|    Aarav Sharma|  Male|  20|Computer Science|   2|3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       104|        Sita Rai|Female|  19|Computer Science|   1|3.9|        Yes|Kathmandu|    sita.rai@uni.edu|
|       108|     Kavya Joshi|Female|NULL|Computer Science|   2|3.7|        Yes|  Pokhara| kavya.joshi@uni.edu|
|       112|     Manisha Oli|Female|  21|Computer Science|   3|3.6|        Yes|  Pokhara| manisha.oli@uni.edu|
|       120|Kabita Bhattarai|Female|  22|Computer Science|   4|3.9|        Yes|Bhaktapur|kabita.bhattarai@...|
+----------+----------------+------+----+----------------+----+---+-----------+---------+--------------------+



In [35]:
df.filter((col("year").isin(3, 4)) & (col("scholarship") == "Yes")).show()

+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|student_id|            name|gender|age|      department|year|gpa|scholarship|     city|               email|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|       111|   Prakash Bista|  Male| 24|        Business|   4|3.4|        Yes|Kathmandu|prakash.bista@uni...|
|       112|     Manisha Oli|Female| 21|Computer Science|   3|3.6|        Yes|  Pokhara| manisha.oli@uni.edu|
|       116|     Rupa Basnet|Female| 23|            NULL|   4|3.7|        Yes|  Pokhara| rupa.basnet@uni.edu|
|       120|Kabita Bhattarai|Female| 22|Computer Science|   4|3.9|        Yes|Bhaktapur|kabita.bhattarai@...|
|       121|      Suman Giri|  Male| 21|     Engineering|   3|3.4|        Yes|Kathmandu|  suman.giri@uni.edu|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+



In [36]:
df = df.withColumn("grade", when(col("gpa") >= 3.7, "Distinction").when(col("gpa") >= 3.0, "Merit").otherwise("Pass"))
df.show()

+----------+----------------+------+----+----------------+----+----+-----------+---------+--------------------+-----------+
|student_id|            name|gender| age|      department|year| gpa|scholarship|     city|               email|      grade|
+----------+----------------+------+----+----------------+----+----+-----------+---------+--------------------+-----------+
|       101|    Aarav Sharma|  Male|  20|Computer Science|   2| 3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|Distinction|
|       102|     Priya Thapa|Female|  22|        Business|   4| 3.2|         No|  Pokhara| priya.thapa@uni.edu|      Merit|
|       103|     Rohan Karki|  Male|  21|     Engineering|   3|NULL|         No| Lalitpur| rohan.karki@uni.edu|       Pass|
|       104|        Sita Rai|Female|  19|Computer Science|   1| 3.9|        Yes|Kathmandu|    sita.rai@uni.edu|Distinction|
|       105|   Bikash Gurung|  Male|  23|        Business|   4| 2.7|         No|   Butwal|bikash.gurung@uni...|       Pass|
|       

## Part D — Dropping & Renaming

In [43]:
df = df.drop("email", "student_id")
print("Columns after dropping:", df.columns)

Columns after dropping: ['name', 'gender', 'age', 'department', 'year', 'gpa', 'scholarship', 'city', 'grade']


In [44]:
df = df.filter(col("department").isNotNull()).dropDuplicates(["name", "department"])
df.show()

+----------------+------+----+----------------+----+----+-----------+---------+-----------+
|            name|gender| age|      department|year| gpa|scholarship|     city|      grade|
+----------------+------+----+----------------+----+----+-----------+---------+-----------+
|    Aarav Sharma|  Male|  20|Computer Science|   2| 3.8|        Yes|Kathmandu|Distinction|
|  Anil Chaudhary|  Male|  21|     Engineering|   3| 2.6|         No|Nepalgunj|       Pass|
|  Anita Shrestha|Female|  20|     Engineering|   2| 3.5|        Yes|Bhaktapur|      Merit|
|   Bikash Gurung|  Male|  23|        Business|   4| 2.7|         No|   Butwal|       Pass|
|  Bishal Ghimire|  Male|  23|     Engineering|   4| 2.8|         No|Kathmandu|       Pass|
|   Deepak Pandey|  Male|  22|            Arts|   3| 3.1|         No|Kathmandu|      Merit|
|  Dipika Niroula|Female|  20|        Business|   2|NULL|         No|   Dharan|       Pass|
|Kabita Bhattarai|Female|  22|Computer Science|   4| 3.9|        Yes|Bhaktapur|D

In [45]:
df = df.withColumnRenamed("gpa", "grade_point_avg").withColumnRenamed("year", "study_year")
df.show()

+----------------+------+----+----------------+----------+---------------+-----------+---------+-----------+
|            name|gender| age|      department|study_year|grade_point_avg|scholarship|     city|      grade|
+----------------+------+----+----------------+----------+---------------+-----------+---------+-----------+
|    Aarav Sharma|  Male|  20|Computer Science|         2|            3.8|        Yes|Kathmandu|Distinction|
|  Anil Chaudhary|  Male|  21|     Engineering|         3|            2.6|         No|Nepalgunj|       Pass|
|  Anita Shrestha|Female|  20|     Engineering|         2|            3.5|        Yes|Bhaktapur|      Merit|
|   Bikash Gurung|  Male|  23|        Business|         4|            2.7|         No|   Butwal|       Pass|
|  Bishal Ghimire|  Male|  23|     Engineering|         4|            2.8|         No|Kathmandu|       Pass|
|   Deepak Pandey|  Male|  22|            Arts|         3|            3.1|         No|Kathmandu|      Merit|
|  Dipika Niroula|F

In [46]:
new_cols = ["full_name","sex","year","dept","year_of_study","gpa","scholarship_status","city_of_residence","level"]
df = df.toDF(*new_cols).show()

+----------------+------+----+----------------+-------------+----+------------------+-----------------+-----------+
|       full_name|   sex|year|            dept|year_of_study| gpa|scholarship_status|city_of_residence|      level|
+----------------+------+----+----------------+-------------+----+------------------+-----------------+-----------+
|    Aarav Sharma|  Male|  20|Computer Science|            2| 3.8|               Yes|        Kathmandu|Distinction|
|  Anil Chaudhary|  Male|  21|     Engineering|            3| 2.6|                No|        Nepalgunj|       Pass|
|  Anita Shrestha|Female|  20|     Engineering|            2| 3.5|               Yes|        Bhaktapur|      Merit|
|   Bikash Gurung|  Male|  23|        Business|            4| 2.7|                No|           Butwal|       Pass|
|  Bishal Ghimire|  Male|  23|     Engineering|            4| 2.8|                No|        Kathmandu|       Pass|
|   Deepak Pandey|  Male|  22|            Arts|            3| 3.1|      

## Part E — Statistics & Aggregation

In [54]:
df = df_filled.dropna()

df.describe("gpa", "age").show()

res = df.agg(F.mean("gpa").alias("mean_gpa"), F.min("age").alias("min_age")).collect()[0]
print("Mean GPA:", float(res["mean_gpa"]))
print("Minimum age:", int(res["min_age"]))

+-------+-------------------+------------------+
|summary|                gpa|               age|
+-------+-------------------+------------------+
|  count|                 25|                25|
|   mean|  3.304545450210571|              21.0|
| stddev|0.36909493967779344|1.3844373104863457|
|    min|                2.6|                19|
|    max|                3.9|                24|
+-------+-------------------+------------------+

Mean GPA: 3.304545450210571
Minimum age: 19


In [55]:
dept_avg_gpa = df_filled.groupBy("department").agg(F.avg("gpa").alias("avg_gpa")).sort(F.col("avg_gpa").desc())
dept_avg_gpa.show()

+----------------+------------------+
|      department|           avg_gpa|
+----------------+------------------+
|      Undeclared| 3.700000047683716|
|Computer Science| 3.614285707473755|
|        Business|3.2209091186523438|
|            Arts|3.2009090423583983|
|     Engineering| 3.072077921458653|
+----------------+------------------+



In [56]:
df_filled.groupBy("gender", "scholarship").agg(F.count("*").alias("count")).show()

+------+-----------+-----+
|gender|scholarship|count|
+------+-----------+-----+
|  Male|         No|   10|
|  Male|        Yes|    3|
|Female|         No|    5|
|Female|        Yes|    7|
+------+-----------+-----+



In [57]:
df_filled.groupBy("city").agg(
    F.count("*").alias("num_students"),
    F.avg("gpa").alias("avg_gpa"),
    F.sum(when(col("scholarship") == "Yes", 1).otherwise(0)).alias("scholarship_count")
).filter(col("num_students") > 2).show()

+---------+------------+------------------+-----------------+
|     city|num_students|           avg_gpa|scholarship_count|
+---------+------------+------------------+-----------------+
|  Pokhara|           5|3.5599999904632567|                4|
| Lalitpur|           4| 3.227272689342499|                0|
|Kathmandu|           8|3.3375000059604645|                4|
+---------+------------+------------------+-----------------+



## Bonus Challenge ★

In [59]:
df_clean = (df
    .filter(~((col("gpa").isNull()) & (col("age").isNull())))
    .join(
        df.groupBy("department").agg(F.avg("gpa").alias("dept_avg_gpa")),
        on="department",
        how="left"
    )
    .withColumn("gpa", when(col("gpa").isNull(), col("dept_avg_gpa")).otherwise(col("gpa")))
    .drop("dept_avg_gpa")
    .withColumn("grade", when(col("gpa") >= 3.7, "Distinction").when(col("gpa") >= 3.0, "Merit").otherwise("Pass"))
    .drop("email")
    .withColumnRenamed("gpa", "grade_point_avg")
    .sort(F.col("grade_point_avg").desc())
)

df_clean.write.mode("overwrite").parquet("output/student_cleaned.parquet")
print("Cleaned dataset saved to output/student_cleaned.parquet")
df_clean.show()

Cleaned dataset saved to output/student_cleaned.parquet
+----------------+----------+----------------+------+---+----+------------------+-----------+---------+-----------+
|      department|student_id|            name|gender|age|year|   grade_point_avg|scholarship|     city|      grade|
+----------------+----------+----------------+------+---+----+------------------+-----------+---------+-----------+
|Computer Science|       104|        Sita Rai|Female| 19|   1|3.9000000953674316|        Yes|Kathmandu|Distinction|
|Computer Science|       120|Kabita Bhattarai|Female| 22|   4|3.9000000953674316|        Yes|Bhaktapur|Distinction|
|Computer Science|       101|    Aarav Sharma|  Male| 20|   2| 3.799999952316284|        Yes|Kathmandu|Distinction|
|Computer Science|       108|     Kavya Joshi|Female| 20|   2| 3.700000047683716|        Yes|  Pokhara|Distinction|
|      Undeclared|       116|     Rupa Basnet|Female| 23|   4| 3.700000047683716|        Yes|  Pokhara|Distinction|
|Computer Scienc